# 📦 Stockout & Overstock Risk Profiling
### UCI Online Retail II Dataset

**Author:** Rahmadhania  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn  
**Dataset:** [UCI Online Retail II via Kaggle](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)

---

## 🎯 Business Question
> **Which products show high sales velocity but low transaction frequency (stockout risk),  
> and which show the opposite — low velocity but high frequency (overstock risk)?**

Inventory imbalance costs retailers in two ways:
- **Stockouts** → lost sales, unhappy customers, damaged brand trust
- **Overstock** → tied-up capital, higher storage cost, potential write-offs

This analysis uses **ABC segmentation** and **sales velocity metrics** to profile every SKU in the catalogue and flag those that need immediate attention.

---

## 📋 Analytical Flow
```
Load Data → Clean Data → ABC Segmentation → Velocity & Frequency → Risk Scoring → Visualization → Recommendation
```

## 0. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

OUTPUT_DIR = '../outputs/charts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

COLORS = {
    'A': '#E63946', 'B': '#F4A261', 'C': '#2A9D8F',
    'stockout': '#E63946', 'overstock': '#457B9D',
    'balanced': '#2A9D8F', 'watch': '#F4A261',
}

print('✓ Environment ready')

---
## 1. Load Data

The dataset has **two sheets** — one per year. We concatenate them into a single DataFrame.

In [ ]:
df_09 = pd.read_excel('../data/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_10 = pd.read_excel('../data/online_retail_II.xlsx', sheet_name='Year 2010-2011')

df = pd.concat([df_09, df_10], ignore_index=True)

print(f'Shape     : {df.shape}')
print(f'Columns   : {list(df.columns)}')
print(f'Date range: {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
df.head(3)

In [ ]:
# Quick snapshot of data quality before cleaning
print('Null counts per column:')
print(df.isnull().sum())
print(f'\nNegative Quantity rows: {(df["Quantity"] < 0).sum():,}')
print(f'Zero/negative Price rows: {(df["Price"] <= 0).sum():,}')

---
## 2. Data Cleaning

| Issue | Action | Reason |
|---|---|---|
| Invoice starts with "C" | Remove | Cancellation, not a sale |
| Non-product StockCodes | Remove | Admin/service charges |
| Null Customer ID | Remove | Cannot attribute demand |
| Quantity ≤ 0 | Remove | Returns or data errors |
| Price ≤ 0 | Remove | Samples or internal use |
| Duplicates | Remove | Prevent double-counting |

In [ ]:
raw_count = len(df)

# Remove cancellations
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Remove non-product StockCodes
non_product = ['POST', 'D', 'M', 'BANK CHARGES', 'CRUK', 'DOT', 'AMAZONFEE', 'B', 'S', 'DCGSSBOY', 'DCGSSGIRL']
df = df[~df['StockCode'].astype(str).isin(non_product)]
df = df[~df['StockCode'].astype(str).str.startswith('gift')]

# Remove null Customer ID
df = df.dropna(subset=['Customer ID'])

# Remove bad Quantity / Price
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

# Remove duplicates
df = df.drop_duplicates()

# Derived columns
df['Revenue'] = df['Quantity'] * df['Price']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')

print(f'Rows removed : {raw_count - len(df):,}')
print(f'Rows retained: {len(df):,}')
print(f'Unique SKUs  : {df["StockCode"].nunique():,}')

---
## 3. ABC Segmentation

**ABC analysis** is a standard inventory management technique based on the **Pareto principle**:

| Class | Cumulative Revenue | Interpretation |
|---|---|---|
| **A** | 0–80% | High-value SKUs — protect at all costs |
| **B** | 80–95% | Mid-value SKUs — monitor regularly |
| **C** | 95–100% | Low-value SKUs — low priority |

In [ ]:
# Aggregate revenue and quantity per SKU
sku_revenue = (
    df.groupby(['StockCode', 'Description'])
    .agg(
        TotalRevenue=('Revenue', 'sum'),
        TotalQuantity=('Quantity', 'sum'),
        TransactionCount=('Invoice', 'nunique'),
    )
    .reset_index()
    .sort_values('TotalRevenue', ascending=False)
)

# Cumulative revenue percentage
sku_revenue['CumulativePct'] = (
    sku_revenue['TotalRevenue'].cumsum() / sku_revenue['TotalRevenue'].sum() * 100
)

# Assign ABC class
def assign_abc(cum_pct):
    if cum_pct <= 80: return 'A'
    elif cum_pct <= 95: return 'B'
    else: return 'C'

sku_revenue['ABC_Class'] = sku_revenue['CumulativePct'].apply(assign_abc)

# Summary table
abc_summary = sku_revenue.groupby('ABC_Class').agg(
    SKU_Count=('StockCode', 'count'),
    Total_Revenue=('TotalRevenue', 'sum'),
).reset_index()
abc_summary['Revenue_Pct'] = (abc_summary['Total_Revenue'] / abc_summary['Total_Revenue'].sum() * 100).round(1)
abc_summary['SKU_Pct'] = (abc_summary['SKU_Count'] / abc_summary['SKU_Count'].sum() * 100).round(1)

abc_summary

In [ ]:
# ── Chart 1: ABC Pareto Curve ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

color_map = sku_revenue['ABC_Class'].map(COLORS)
ax.bar(range(len(sku_revenue)), sku_revenue['TotalRevenue'], color=color_map, width=1.0, linewidth=0)

ax2 = ax.twinx()
ax2.plot(range(len(sku_revenue)), sku_revenue['CumulativePct'], color='#1D3557', linewidth=2)
ax2.axhline(80, color=COLORS['A'], linestyle='--', linewidth=1, alpha=0.7)
ax2.axhline(95, color=COLORS['B'], linestyle='--', linewidth=1, alpha=0.7)
ax2.set_ylabel('Cumulative Revenue %', color='#1D3557')
ax2.set_ylim(0, 105)

patches = [
    mpatches.Patch(color=COLORS['A'], label='Class A (80% revenue)'),
    mpatches.Patch(color=COLORS['B'], label='Class B (next 15%)'),
    mpatches.Patch(color=COLORS['C'], label='Class C (bottom 5%)'),
]
ax.legend(handles=patches, loc='upper left', fontsize=9)
ax.set_title('ABC Segmentation — Revenue Pareto Curve', fontweight='bold', pad=12)
ax.set_xlabel('SKUs (sorted by revenue, high → low)')
ax.set_ylabel('Revenue (GBP)')
ax.set_xticks([])

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_abc_pareto_curve.png')
plt.show()

In [ ]:
# ── Chart 2: ABC SKU Count vs Revenue Share ───────────────────────────────────
classes = ['A', 'B', 'C']
sku_pcts = [abc_summary.loc[abc_summary['ABC_Class']==c, 'SKU_Pct'].values[0] for c in classes]
rev_pcts = [abc_summary.loc[abc_summary['ABC_Class']==c, 'Revenue_Pct'].values[0] for c in classes]
bar_colors = [COLORS[c] for c in classes]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(classes, sku_pcts, color=bar_colors, width=0.5)
axes[0].set_title('% of SKUs by Class', fontweight='bold')
axes[0].set_ylabel('% of Total SKUs')
for i, v in enumerate(sku_pcts):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)

axes[1].bar(classes, rev_pcts, color=bar_colors, width=0.5)
axes[1].set_title('% of Revenue by Class', fontweight='bold')
axes[1].set_ylabel('% of Total Revenue')
for i, v in enumerate(rev_pcts):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)

fig.suptitle('ABC Class Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_abc_distribution.png', bbox_inches='tight')
plt.show()

---
## 4. Sales Velocity & Transaction Frequency

Two metrics drive the risk model:

- **Sales Velocity** = Total Quantity ÷ Active Months  
  → *How fast does this product move?*

- **Transaction Frequency** = Invoice Count ÷ Active Months  
  → *How often is this product ordered?*

A product can sell a lot in volume (high velocity) but only appear in orders a few times a year (low frequency) — that's the classic **bulk-order pattern** that creates stockout gaps between shipments.

In [ ]:
# Active months per SKU = number of distinct YearMonth periods it appeared in
sku_activity = (
    df.groupby('StockCode')['YearMonth']
    .nunique()
    .reset_index()
    .rename(columns={'YearMonth': 'ActiveMonths'})
)

sku_revenue = sku_revenue.merge(sku_activity, on='StockCode', how='left')

# Velocity and frequency
sku_revenue['Velocity']  = sku_revenue['TotalQuantity']    / sku_revenue['ActiveMonths']
sku_revenue['Frequency'] = sku_revenue['TransactionCount'] / sku_revenue['ActiveMonths']

print('Sales Velocity (units/month):')
print(sku_revenue['Velocity'].describe().round(2))
print('\nTransaction Frequency (orders/month):')
print(sku_revenue['Frequency'].describe().round(2))

---
## 5. Risk Scoring

We split the SKU catalogue into four quadrants using the **median** of velocity and frequency as thresholds:

| Quadrant | Velocity | Frequency | Interpretation |
|---|---|---|---|
| 🔴 **Stockout Risk** | High | Low | Fast-moving, rarely reordered → gaps likely |
| 🔵 **Overstock Risk** | Low | High | Slow-moving, frequently ordered → capital tied up |
| 🟢 **Balanced** | High | High | Healthy — fast and frequently restocked |
| 🟡 **Watch** | Low | Low | Slow and infrequent — potential dead stock |

In [ ]:
v_median = sku_revenue['Velocity'].median()
f_median = sku_revenue['Frequency'].median()

print(f'Velocity  median: {v_median:.2f} units/month')
print(f'Frequency median: {f_median:.2f} transactions/month')

def assign_risk(row):
    hi_v = row['Velocity']  >= v_median
    hi_f = row['Frequency'] >= f_median
    if   hi_v and not hi_f: return 'Stockout Risk'
    elif not hi_v and hi_f: return 'Overstock Risk'
    elif hi_v and hi_f:     return 'Balanced'
    else:                   return 'Watch'

sku_revenue['RiskCategory'] = sku_revenue.apply(assign_risk, axis=1)

risk_summary = (
    sku_revenue.groupby('RiskCategory')
    .agg(SKU_Count=('StockCode','count'), Avg_Revenue=('TotalRevenue','mean'))
    .reset_index()
    .sort_values('SKU_Count', ascending=False)
)
risk_summary

---
## 6. Visualizations

In [ ]:
# ── Chart 3: Risk Matrix Scatter ──────────────────────────────────────────────
risk_color_map = {
    'Stockout Risk': COLORS['stockout'], 'Overstock Risk': COLORS['overstock'],
    'Balanced': COLORS['balanced'],      'Watch': COLORS['watch'],
}

v_cap = sku_revenue['Velocity'].quantile(0.99)
f_cap = sku_revenue['Frequency'].quantile(0.99)
plot_df = sku_revenue[(sku_revenue['Velocity'] <= v_cap) & (sku_revenue['Frequency'] <= f_cap)].copy()

fig, ax = plt.subplots(figsize=(10, 7))

for risk, group in plot_df.groupby('RiskCategory'):
    ax.scatter(group['Velocity'], group['Frequency'],
               c=risk_color_map[risk], label=risk, alpha=0.5, s=20, linewidths=0)

ax.axvline(v_median, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.axhline(f_median, color='gray', linestyle='--', linewidth=1, alpha=0.6)

ax.text(v_cap*0.95, f_median*0.1,  'WATCH',          ha='right', color=COLORS['watch'],     fontsize=9, fontstyle='italic')
ax.text(v_cap*0.95, f_cap*0.95,    'BALANCED',        ha='right', color=COLORS['balanced'],   fontsize=9, fontstyle='italic')
ax.text(v_median*0.05, f_cap*0.95, 'OVERSTOCK RISK',  ha='left',  color=COLORS['overstock'],  fontsize=9, fontstyle='italic')
ax.text(v_median*0.05, f_median*0.1,'WATCH',          ha='left',  color=COLORS['watch'],     fontsize=9, fontstyle='italic')

ax.set_title('Inventory Risk Matrix — Velocity vs. Transaction Frequency', fontweight='bold', pad=12)
ax.set_xlabel('Sales Velocity (units/month)')
ax.set_ylabel('Transaction Frequency (orders/month)')
ax.legend(loc='upper center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.08), frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_risk_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 4: Risk by ABC Class ────────────────────────────────────────────────
risk_abc = (
    sku_revenue.groupby(['ABC_Class', 'RiskCategory'])
    .size().unstack(fill_value=0)
    .reindex(['A', 'B', 'C'])
)
risk_abc_pct = risk_abc.div(risk_abc.sum(axis=1), axis=0) * 100
risk_plot_colors = [risk_color_map.get(c, '#999') for c in risk_abc_pct.columns]

fig, ax = plt.subplots(figsize=(9, 5))
risk_abc_pct.plot(kind='bar', stacked=True, color=risk_plot_colors, ax=ax, width=0.5,
                  edgecolor='white', linewidth=0.5)
ax.set_title('Risk Profile by ABC Class (% of SKUs)', fontweight='bold', pad=12)
ax.set_xlabel('ABC Class')
ax.set_ylabel('% of SKUs in Class')
ax.set_xticklabels(['A (High Value)', 'B (Mid Value)', 'C (Low Value)'], rotation=0)
ax.legend(loc='upper right', fontsize=9, frameon=False)
ax.set_ylim(0, 115)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_risk_by_abc_class.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 5: Top 15 Stockout Risk SKUs ───────────────────────────────────────
top_stockout = (
    sku_revenue[sku_revenue['RiskCategory'] == 'Stockout Risk']
    .sort_values('TotalRevenue', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_stockout['Description'].str[:40], top_stockout['TotalRevenue'],
               color=COLORS['stockout'], alpha=0.85)
ax.set_title('Top 15 High-Revenue SKUs at Stockout Risk', fontweight='bold', pad=12)
ax.set_xlabel('Total Revenue (GBP)')
ax.invert_yaxis()
for bar, val in zip(bars, top_stockout['TotalRevenue']):
    ax.text(bar.get_width()+200, bar.get_y()+bar.get_height()/2,
            f'£{val:,.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_top_stockout_skus.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 6: Monthly Revenue Trend ───────────────────────────────────────────
monthly = df.groupby('YearMonth')['Revenue'].sum().reset_index()
monthly['YearMonth_str'] = monthly['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(range(len(monthly)), monthly['Revenue'], alpha=0.25, color='#1D3557')
ax.plot(range(len(monthly)), monthly['Revenue'], color='#1D3557', linewidth=2)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['YearMonth_str'], rotation=45, ha='right', fontsize=8)
ax.set_title('Monthly Revenue Trend (Dec 2009 – Dec 2011)', fontweight='bold', pad=12)
ax.set_ylabel('Revenue (GBP)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_monthly_revenue_trend.png', bbox_inches='tight')
plt.show()

---
## 7. Export Results

In [ ]:
# Full SKU risk profile
sku_revenue.to_csv('../outputs/sku_risk_profile.csv', index=False)

# Stockout priority list
stockout_list = (
    sku_revenue[sku_revenue['RiskCategory'] == 'Stockout Risk']
    [['StockCode','Description','ABC_Class','TotalRevenue','TotalQuantity',
      'TransactionCount','ActiveMonths','Velocity','Frequency']]
    .sort_values(['ABC_Class','TotalRevenue'], ascending=[True,False])
)
stockout_list.to_csv('../outputs/stockout_priority_list.csv', index=False)

# Overstock priority list
overstock_list = (
    sku_revenue[sku_revenue['RiskCategory'] == 'Overstock Risk']
    [['StockCode','Description','ABC_Class','TotalRevenue','TotalQuantity',
      'TransactionCount','ActiveMonths','Velocity','Frequency']]
    .sort_values(['ABC_Class','TotalRevenue'], ascending=[True,False])
)
overstock_list.to_csv('../outputs/overstock_priority_list.csv', index=False)

print('✓ All outputs exported to outputs/')

---
## 8. Key Findings & Recommendations

### 📊 Summary

In [ ]:
total_skus = len(sku_revenue)
print('=' * 50)
print('RISK CATEGORY BREAKDOWN')
print('=' * 50)
for cat in ['Stockout Risk', 'Overstock Risk', 'Balanced', 'Watch']:
    count = len(sku_revenue[sku_revenue['RiskCategory'] == cat])
    pct = count / total_skus * 100
    print(f'  {cat:<18}: {count:>5} SKUs ({pct:.1f}%)')

a_stockout = len(sku_revenue[(sku_revenue['ABC_Class']=='A') & (sku_revenue['RiskCategory']=='Stockout Risk')])
a_total    = len(sku_revenue[sku_revenue['ABC_Class']=='A'])
print(f'\n⚠  Class-A SKUs at Stockout Risk: {a_stockout} of {a_total} ({a_stockout/a_total*100:.1f}%)')

### 💡 Business Recommendations

**1. Prioritise Class-A Stockout-Risk SKUs for reorder point review**  
These are high-revenue products that sell fast but are ordered infrequently. A single stockout event on a Class-A item can represent thousands of GBP in lost revenue. Review reorder points and set safety stock levels immediately.

**2. Audit Overstock-Risk SKUs for order frequency reduction**  
Products in the Overstock Risk quadrant are being reordered too frequently relative to their actual sales velocity. Work with procurement to extend reorder cycles or reduce batch sizes to free up working capital.

**3. Flag 'Watch' SKUs for potential discontinuation**  
Low velocity + low frequency = potential dead stock. Evaluate whether these SKUs have seasonal relevance or should be phased out to reduce catalogue complexity and storage costs.

**4. Use ABC Class as a service-level tier**  
Set different service-level targets per class: 99%+ availability for Class A, 95% for Class B, best-effort for Class C. This prevents over-investing in low-value SKUs.

**5. Investigate the November revenue spike**  
The monthly trend shows a consistent revenue spike in November (pre-holiday demand). Ensure Class-A Stockout-Risk SKUs are over-stocked ahead of October each year to capture this demand.

---
*Analysis complete — see `outputs/` for exportable CSVs and `outputs/charts/` for all visualizations.*